Prepare environment and data as specified in [README](https://github.com/njukenanli/ChemBart_Planner)

In [ ]:
product = "BrCC(Br)COC(=O)CCCCCCNC(=O)c1ccccc1"
reagent = "S(=O)(=O)(O)(O)"
reactant = "BrCC(Br)CO.OC(=O)CCCCCCNC(=O)c1ccccc1"
precursor_list = [
    "BrCC(Br)CO.OC(=O)CCCCCCNC(=O)c1ccccc1"
    "C=CCOC(=O)CCCCCCNC(=O)c1ccccc1.BrBr"
    "BrCC(Br)COC(=O)CCCCCCN.OC(=O)c1ccccc1"
    "BrCC(Br)CO.OC(=O)CCCCCCN.OC(=O)c1ccccc1"
    "BrCC(Br)CO.C1(=O)CCCCCCN1.OC(=O)c1ccccc1"
]

import os
os.chdir("../")

In [ ]:
from api import CBRetro
reaction_model = CBRetro(path = "model/Pretrained-Full.pth", dev = "cuda:0")
reactant_candidates = reaction_model.precursor(
    product = product,
    top_k = 10,
    top_p = 0.9,
    sampling_method = "top_p" # or top_k, beam
    num_samples = 10
) 
# top_k means top_k tokens at each auto_regressive token generation step would be retained
# top_k is not used when sampling_method = "beam"
# num_samples means top num_samples final answers would be returned

reagent_candidates = reaction_model.reagent(
    reactant = reactant,
    product = product,
    k = 10,
)

product_candidates = reaction_model.product(
    reactant = reactant,
    reagent = reagent,
    k = 10,
)


In [ ]:
from api import CBRL

policy_value_model = CBRL(path = "model/Policy-Value.pth", dev = "cuda:0")

policy = policy_value_model.policy(product=product, precursorlist=precursor_list)

value = policy_value_model.value(product=product)

In [ ]:
from api import CBTempYield

temperature_yield_model = CBTempYield(path = "model/Temperature-Yield.pth", dev = "cuda:0")

temperature, yield = temperature_yield_model.pred(reactant, reagant, product)